## Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import geopandas as gp
import shapely as sh
import re
import os
from tqdm.auto import tqdm
from pathlib import Path
tqdm.pandas()

# Define Paths relative to project root
PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / 'Data' / 'Raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'Processed'

# Target 'north' coverage and relevant counties
coverage = 'north'
north_list = ['NORTHUMBERLAND', 'CUMBERLAND', 'DURHAM', 'WESTMORLAND',
              'YORKSHIRE, NORTH RIDING', 'LANCASHIRE', 'YORKSHIRE, WEST RIDING',
              'YORKSHIRE, EAST RIDING', 'LINCOLNSHIRE']

c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Valor Dataset Creation (Logic from 00_Valor_dataset_creator.py)

In [2]:
print('--- Step 1: Loading and Processing Valor data ---')
line_items_df = pd.read_csv(RAW / 'CSV/ValorLineItems.csv')
line_items_df = line_items_df[line_items_df['multi'] != 1]
natl_archives_df = pd.read_csv(RAW / 'CSV/NationalArchivesData.csv')

# Load spatial buffers for urban/suburban classification
suburban_buffer_gdf = gp.read_file(RAW / 'GIS/BNG Projections/SuburbanBufferBNG.shp')
urban_buffer_gdf = gp.read_file(RAW / 'GIS/BNG Projections/UrbanBufferBNG1.shp')

# Merge and clean data
df = pd.merge(line_items_df, natl_archives_df, on='name', validate='m:1', how='right')
df = df[df['note'].str.contains('italic') != True]
df = df.drop(columns=['note', 'page', 'foundedSource'])
df['total'] = pd.to_numeric(df['total'], errors='coerce')
df = df.dropna(subset=['total'])

# Categorization Regex
catVars = ['land', 'tithe', 'mill', 'fee', 'alms', 'infra', 'transfer', 'court', 'annuity', 'education', 'synProx', 'unknown']
synProx = re.compile('(syn)|(prox)')
glebe = re.compile('(glebe)')
ownLand = re.compile('(own land)')

def get_categories(row):
    """Categorize items based on transfer type and party."""
    if (row['transfer'] == 1) and (re.search(synProx, row['counterParty']) is not None):
        row['synProx'] = 1
        row['transfer'] = 0
    else:
        row['synProx'] = 0
    if (row['land'] == 1) and (re.search(glebe, str(row['counterParty'])) is not None):
        row['tithe'] = 1
        row['land'] = 0
    row['incomplete'] = 1 if len(df[df['name'] == row['name']]) <= 3 else 0
    row['ladyHouse'] = 1 if row['members'] in (['Nuns', 'Canonesses']) else 0
    row['ownLand'] = 1 if row['land'] == 1 and re.search(ownLand, str(row['counterParty']).lower()) is not None else 0
    return row

df = df.progress_apply(get_categories, axis=1)
df[catVars + ['sum']] = df[catVars + ['sum']].fillna(0).apply(pd.to_numeric, errors='coerce')
df[['lat', 'long', 'latitude', 'longitude']] = df[['lat', 'long', 'latitude', 'longitude']].apply(pd.to_numeric, errors='coerce')

# Assign primary type
for var in catVars:
    df.loc[df[var] == 1, 'type'] = var
df['type'] = df['type'].fillna('sum')

--- Step 1: Loading and Processing Valor data ---


100%|██████████| 18082/18082 [00:33<00:00, 543.21it/s]


## Create Lines and Flows

In [3]:
print('Creating geographic lines for flows...')
lines_gdf = df.dropna(subset=['lat', 'long']).copy()
income_df = lines_gdf[lines_gdf['total'] > 0].copy()
expenditure_df = lines_gdf[lines_gdf['total'] < 0].copy()

# Create LineString geometries based on flow direction
income_df['lineString'] = income_df.apply(lambda i: sh.geometry.LineString([(i['long'], i['lat']), (i['longitude'], i['latitude'])]), axis=1)
income_df['direction'] = 'in'
expenditure_df['lineString'] = expenditure_df.apply(lambda e: sh.geometry.LineString([(e['longitude'], e['latitude']), (e['long'], e['lat'])]), axis=1)
expenditure_df['direction'] = 'out'

lines_gdf = pd.concat([income_df, expenditure_df])
lines_gdf['total'] = abs(lines_gdf['total'])
lines_gdf = gp.GeoDataFrame(lines_gdf, geometry='lineString', crs='epsg:4326').to_crs('epsg:27700')
lines_gdf = lines_gdf.rename(columns={'lineString': 'geometry'}).set_geometry('geometry')

# Add smallHouse flag early for line processing
# First calculate total income per house
house_totals = df[df['total'] > 0].groupby('name')['total'].sum().rename('totalInc')
lines_gdf = lines_gdf.merge(house_totals, on='name', how='left')
lines_gdf['smallHouse'] = (lines_gdf['totalInc'] <= (200 * 240)).astype(int)

def get_urban_status(row):
    """Classify flow source/destination as Urban, Suburban, or Rural."""
    pt0 = sh.geometry.Point(row['geometry'].coords[0])
    pt1 = sh.geometry.Point(row['geometry'].coords[1])
    sub_src = suburban_buffer_gdf['geometry'].contains(pt0).any()
    sub_dest = suburban_buffer_gdf['geometry'].contains(pt1).any()
    urb_src = urban_buffer_gdf['geometry'].contains(pt0).any()
    urb_dest = urban_buffer_gdf['geometry'].contains(pt1).any()
    
    row['urbSrc'], row['subSrc'] = (1, 0) if urb_src else (0, 1 if sub_src else 0)
    row['urbDest'], row['subDest'] = (1, 0) if urb_dest else (0, 1 if sub_dest else 0)
    row['rurSrc'] = 1 if row['urbSrc'] == 0 and row['subSrc'] == 0 else 0
    row['rurDest'] = 1 if row['urbDest'] == 0 and row['subDest'] == 0 else 0
    
    if row['direction'] == 'out':
        row['urbHouse'], row['subHouse'], row['rurHouse'] = row['urbSrc'], row['subSrc'], row['rurSrc']
    else:
        row['urbHouse'], row['subHouse'], row['rurHouse'] = row['urbDest'], row['subDest'], row['rurDest']
    return row

print('Classifying urban status for lines...')
lines_gdf = lines_gdf.progress_apply(get_urban_status, axis=1)
lines_gdf['diss1536'] = (lines_gdf['year'] == 1536).astype(int)
north_lines_gdf = lines_gdf[lines_gdf['county'].isin(north_list)].copy()

Creating geographic lines for flows...
Classifying urban status for lines...


100%|██████████| 12446/12446 [00:29<00:00, 424.86it/s]


## 2. Parish Shapefile Processing

In [4]:
print('--- Step 2: Processing Parish Shapefile ---')
parish_df = gp.read_file(RAW / 'GIS/BNG Projections/AncientParishesBNG.shp')
parish_df = parish_df[parish_df['GAZ_CNTY'].isin(north_list)]

# Load auxiliary datasets
lincs_rebel_df = pd.read_csv(RAW / 'CSV/LincsRebels.csv')
muster_df = gp.read_file(RAW / 'GIS/BNG Projections/rebPoints.shp')
muster_df['muster'] = 1
seat_df = gp.read_file(RAW / 'GIS/BNG Projections/gentlemenInvolved.shp')
seat_df['seats'] = 1
terrain_df = gp.read_file(RAW / 'GIS/BNG Projections/TerrainZones.shp')
population_df = gp.read_file(RAW / 'GIS/BNG Projections/CombinedPop.shp')

sheail_payers_df = gp.read_file(RAW / 'GIS/BNG Projections/SheailParishPops1525ND.shp')
sheail_payers_df = sheail_payers_df.drop(columns=['FID']).dissolve(by='taxpayersH').reset_index()
sheail_payers_df = gp.clip(sheail_payers_df, parish_df)

sheail_shillings_df = gp.read_file(RAW / 'GIS/BNG Projections/SheailParishShillings1525ND.shp')
sheail_shillings_df = sheail_shillings_df.drop(columns=['FID', 'layer', 'path']).dissolve(by='shilH').reset_index()
sheail_shillings_df = sheail_shillings_df.rename(columns={'shilH':'shillH', 'shilL':'shillL'})
sheail_shillings_df = gp.clip(sheail_shillings_df, parish_df)

print('Processing points for parish aggregation...')
# Small House categorization
net_df = north_lines_gdf[north_lines_gdf['counterParty']=='Net income'][['total', 'geometry']].rename(columns={'total':'rhNetInc'})
net_df['smHouse'] = (net_df['rhNetInc'] <= (200*240)).astype(int)
net_df['bigHouse'] = (net_df['rhNetInc'] > (200*240)).astype(int)

# Separate flows into source/destination points
lines_proc = north_lines_gdf[north_lines_gdf['sum']==0].copy()

# Distance-decay weights: 1 if distance <= 12.5 km, else min(1, 1/(distance_km - 12.5))
# The min() caps weights at 1 for the 12.5-13.5 km range, preventing near-singularity spikes.
# Decay begins (weight < 1) only once distance exceeds 13.5 km.
llen_km = lines_proc.geometry.length / 1000
lines_proc['dist_w'] = np.where(llen_km <= 12.5, 1.0, np.minimum(1.0, 1.0 / (llen_km - 12.5)))

in_parts, out_parts = [], []

for d in ['in', 'out']:
    temp = lines_proc.copy()
    if d == 'in':
        temp['geometry'] = temp['geometry'].map(lambda x: sh.geometry.Point(list(x.coords)[1]))
        temp['inTot'], temp['outTot'] = temp['total'], 0
        for v in catVars:
            temp[v + 'InTot'] = temp[v] * temp['inTot']
            temp[v + 'OutTot'] = 0
        temp['landOwned'] = temp['dissLand'] = temp['ownLandVal'] = temp['otherLandVal'] = temp['smLand'] = 0
        temp['smTithe'] = temp['bgTithe'] = 0
        temp['lo_dw'] = 0
        temp['sl_dw'] = 0
        temp['bl_dw'] = 0
        temp['ti_dw'] = 0
        temp['st_dw'] = 0
        temp['bt_dw'] = 0
        temp['llo_dw'] = 0
        temp['lsl_dw'] = 0
        temp['lbl_dw'] = 0
        temp['lti_dw'] = 0
        temp['lst_dw'] = 0
        temp['lbt_dw'] = 0
        in_parts.append(temp)
    else:
        temp['geometry'] = temp['geometry'].map(lambda x: sh.geometry.Point(list(x.coords)[0]))
        temp['outTot'], temp['inTot'] = temp['total'], 0
        for v in catVars:
            temp[v + 'OutTot'] = temp[v] * temp['outTot']
            temp[v + 'InTot'] = 0
        temp['landOwned'] = temp.loc[temp['direction'] == 'in', 'landOutTot'].fillna(0)
        temp['dissLand'] = temp['diss1536'] * temp['landOwned']
        temp['ownLandVal'] = temp['landOwned'] * temp['ownLand']
        temp['otherLandVal'] = temp['landOwned'] - temp['ownLandVal']
        temp['smLand'] = temp['landOwned'] * temp['smallHouse']
        temp['lo_dw'] = temp['landOwned'] * temp['dist_w']
        temp['sl_dw'] = temp['smLand'] * temp['dist_w']
        temp['bl_dw'] = (temp['landOwned'] - temp['smLand']) * temp['dist_w']
        temp['smTithe'] = temp['titheOutTot'] * temp['smallHouse']
        temp['bgTithe'] = temp['titheOutTot'] * (1 - temp['smallHouse'])
        temp['ti_dw'] = temp['titheOutTot'] * temp['dist_w']
        temp['st_dw'] = temp['smTithe'] * temp['dist_w']
        temp['bt_dw'] = temp['bgTithe'] * temp['dist_w']
        # Log-weighted versions (log first, then distance-weight)
        temp['llo_dw'] = np.log(temp['landOwned'] + 1) * temp['dist_w']
        temp['lsl_dw'] = np.log(temp['smLand'] + 1) * temp['dist_w']
        temp['lbl_dw'] = np.log((temp['landOwned'] - temp['smLand']) + 1) * temp['dist_w']
        temp['lti_dw'] = np.log(temp['titheOutTot'] + 1) * temp['dist_w']
        temp['lst_dw'] = np.log(temp['smTithe'] + 1) * temp['dist_w']
        temp['lbt_dw'] = np.log(temp['bgTithe'] + 1) * temp['dist_w']
        out_parts.append(temp)

in_out_df = gp.GeoDataFrame(pd.concat(in_parts + out_parts), geometry='geometry', crs='epsg:27700')

--- Step 2: Processing Parish Shapefile ---
Processing points for parish aggregation...


## Merge HRV Data

In [5]:
print('Merging HRV historical records...')
HRVPath = RAW / 'CSV/HRV/'
hrv_dfs = [pd.read_csv(HRVPath / f) for f in os.listdir(HRVPath) if f.endswith('.csv')]
big_df = pd.concat(hrv_dfs, axis=1).loc[:, ~pd.concat(hrv_dfs, axis=1).columns.duplicated()]
big_df = big_df.dropna(subset=['pla'])
if 'STARTcounty' in big_df.columns:
    big_df = big_df.rename(columns={'STARTcounty': 'county'})
big_df['hrv_land'] = big_df['lMincome'].apply(np.exp)

parish_df = parish_df.rename(columns={'PLA': 'pla', 'GAZ_CNTY': 'county', 'AREA': 'area'})
parish_df['area'] /= 1_000_000
parish_df = pd.merge(parish_df, big_df, how='left', on=['pla', 'county', 'area'])

# Aggregation Logic
paSum = ['area', 'NrGentry', 'mills', 'copyhold_count_1850', 'copyhold_count', 'NrPatents', 'NrGentry_1400', 'mills_1400', 'copyhold_count_1516', 'hrv_land']
paMean = ['X_COORD', 'Y_COORD', 'perc_catholics_1800', 'ind_share_1831', 'agr_share_1831', 'ind_share', 'agr_share', 'LS_pc_change', 'pc_change1525_1086', 'pc_change1525_1066', 'pc_change1332_1086', 'pc_change1332_1066', 'pc_change1086_1066', 'lLStax_pc', 'LStax_pc_1332', 'agr_share_1370', 'ind_share_1370', 'WheatYield', 'mean_elevation', 'mean_slope', 'wheatsuitability', 'distancetoriver', 'distancetomarkettown', 'distancetoborder', 'distancetolondon', 'distancetocoal', 'latitude', 'longitude']
paFirst = ['PAR1851_ID', 'PAR1851_', 'county', 'PAR', 'hundred']
parish_df['par_county'] = parish_df['PAR'] + '_' + parish_df['county']
aggDict = {v: 'sum' for v in paSum} | {v: 'mean' for v in paMean} | {v: 'first' for v in paFirst}
parish_df = parish_df.dissolve(by='par_county', aggfunc=aggDict)

Merging HRV historical records...


## Spatial Aggregation of Flows, Musters, and Seats

In [6]:
print('Spatially joining flows and rebellion data...')
io_vars = ['outTot', 'inTot', 'landOwned', 'dissLand', 'smLand', 'ownLandVal', 'otherLandVal', 'smTithe', 'bgTithe',
           'lo_dw', 'sl_dw', 'bl_dw', 'ti_dw', 'st_dw', 'bt_dw', 'llo_dw', 'lsl_dw', 'lbl_dw', 'lti_dw', 'lst_dw', 'lbt_dw'] + [v + 'InTot' for v in catVars] + [v + 'OutTot' for v in catVars]
# Using intersects to ensure points on boundaries are captured
joined_io = gp.sjoin(in_out_df, parish_df[['geometry']], how='right', predicate='intersects').groupby('par_county').agg({v: 'sum' for v in io_vars})
parish_df = parish_df.join(joined_io)

muster_df[['muster', 'day', 'primary']] = muster_df[['muster', 'day', 'primary']].apply(pd.to_numeric, errors='coerce')
joined_muster = gp.sjoin(muster_df, parish_df[['geometry']], how='right', predicate='intersects').groupby('par_county').agg({'muster':'sum', 'day':'mean', 'primary':'max'})
joined_muster['muster'] = (joined_muster['muster'] >= 1).astype(int)
parish_df = parish_df.join(joined_muster)
parish_df['muster'] = parish_df['muster'].fillna(0).astype(int)
parish_df['primary'] = parish_df['primary'].fillna(0).astype(int)

joined_seats = gp.sjoin(seat_df, parish_df[['geometry']], how='right', predicate='intersects').groupby('par_county').agg({'seats':'sum'}).fillna(0)
parish_df = parish_df.join(joined_seats)

# rhNetInc total via spatial intersection (unchanged behaviour)
joined_net = gp.sjoin(net_df, parish_df[['geometry']], how='right', predicate='intersects').groupby('par_county').agg({'rhNetInc':'sum'}).fillna(0)
parish_df = parish_df.join(joined_net)

# --- House-size proximity variables ---
# Mirrors the monastic-opposition pattern in jn_06:
#   smHouse / bigHouse    : 1 if parish centroid within 20 km of any small/big house
#   smHouse_w / bigHouse_w: IDW-weighted exposure; w = 1 if d <= 10 km, else 10 / d_km,
#                           summed across all houses in the group.
# net_df carries LineString geometry (from the flows table); the 'Net income' rows are
# degenerate lines anchored at the house location, so .centroid returns the house point.
BUFFER_M = 20_000
FLAT_RADIUS_M = 10_000

net_points = net_df.copy()
net_points['geometry'] = net_points.geometry.centroid

parish_centroids = parish_df.geometry.centroid
cx_hs = parish_centroids.x.values
cy_hs = parish_centroids.y.values

hs_groups = {
    'smHouse':  net_points['smHouse']  == 1,
    'bigHouse': net_points['bigHouse'] == 1,
}

# 20 km binary proximity
for out_col, mask in hs_groups.items():
    subset = net_points[mask]
    if len(subset) == 0:
        parish_df[out_col] = 0
        print(f"{out_col}: 0 houses -> 0 parishes flagged")
        continue
    buf = subset.geometry.buffer(BUFFER_M).union_all()
    parish_df[out_col] = parish_centroids.within(buf).astype(int)
    print(f"{out_col}: {len(subset)} houses -> {int(parish_df[out_col].sum())} parishes within 20 km")

# IDW exposure
for out_col_base, mask in hs_groups.items():
    out_col = out_col_base + '_w'
    subset = net_points[mask]
    if len(subset) == 0:
        parish_df[out_col] = 0.0
        print(f"{out_col}: 0 houses -> all zeros")
        continue
    mx = subset.geometry.x.values
    my = subset.geometry.y.values
    dist_m = np.sqrt(
        (cx_hs[:, np.newaxis] - mx[np.newaxis, :]) ** 2
        + (cy_hs[:, np.newaxis] - my[np.newaxis, :]) ** 2
    )
    weights = np.where(dist_m <= FLAT_RADIUS_M, 1.0, FLAT_RADIUS_M / dist_m)
    parish_df[out_col] = weights.sum(axis=1)
    col = parish_df[out_col]
    print(f"{out_col}: {len(subset)} houses  min={col.min():.3f}  mean={col.mean():.3f}  max={col.max():.3f}")

parish_df['netFlow'] = parish_df['inTot'] - parish_df['outTot']

Spatially joining flows and rebellion data...
smHouse: 113 houses -> 1652 parishes within 20 km
bigHouse: 36 houses -> 1241 parishes within 20 km
smHouse_w: 113 houses  min=7.028  mean=16.701  max=23.118
bigHouse_w: 36 houses  min=1.719  mean=5.522  max=8.780


## Final Data Adding and Cleaning

In [7]:
print('Adding final variables...')


# Assign Rebellion dummy for Lincolnshire parishes based on LincsRebels.csv
parish_df['reb'] = 0
parNums = dict(zip(parish_df['PAR1851_ID'], parish_df.index))
for pid in lincs_rebel_df['PAR1851_ID']:
    if pid in parNums:
        parish_df.at[parNums[pid], 'reb'] = 1

# Join Friaries
friardf = gp.read_file(RAW / 'GIS/BNG Projections/friarPoints.shp').to_crs('epsg:27700')
friardf['friary'] = 1
joined_friar = gp.sjoin(parish_df[['geometry']], friardf[['geometry', 'friary']], how='left', predicate='contains').groupby('par_county').agg({'friary':'sum'}).fillna(0)
parish_df = parish_df.join(joined_friar)

# Join Terrain Type
print('Joining terrain data...')
# terrain_df was loaded earlier
joined_terrain = gp.sjoin(parish_df[['geometry']], terrain_df, how='left', predicate='intersects')
# Take the first terrain type if a parish overlaps multiple zones
joined_terrain = joined_terrain.groupby('par_county').first()

# Map T_TYPE to terrainTyp as found in the data
if 'T_TYPE' in joined_terrain.columns:
    joined_terrain = joined_terrain.rename(columns={'T_TYPE': 'terrainTyp'})

if 'terrainTyp' in joined_terrain.columns:
    parish_df = parish_df.join(joined_terrain[['terrainTyp']])
else:
    print(f"Warning: 'terrainTyp' not found in terrain_df columns: {joined_terrain.columns.tolist()}")
    # Create dummy column to prevent crash if absolutely missing
    parish_df['terrainTyp'] = 'Unknown'

# Add population data
print('Joining population data...')
# population_df was loaded earlier
population_df = population_df.rename(columns={'pop': 'popC'})
joined_pop = gp.sjoin(parish_df[['geometry']], population_df[['geometry', 'popC']], how='left', predicate='intersects').groupby('par_county').agg({'popC':'sum'}).fillna(0)
parish_df = parish_df.join(joined_pop)

# Distance to Scottish border
# Use the northern boundary of the consolidated north counties as an approximation
print('Calculating distance to Scottish border...')
counties_gdf = gp.read_file(RAW / 'GIS/BNG Projections/CountiesConsolidatedBNG.shp')
name_col = next((c for c in counties_gdf.columns
                 if c.upper() in ['NAME', 'CTYNAME', 'GAZ_CNTY', 'COUNTY', 'COUN_NAME']), None)
if name_col:
    north_cty = counties_gdf[counties_gdf[name_col].str.upper().str.strip().isin(
        [n.upper() for n in north_list])]
    north_union = north_cty.union_all() if len(north_cty) > 0 else parish_df.union_all()
else:
    north_union = parish_df.union_all()
bds = north_union.bounds  # (minx, miny, maxx, maxy)
northing_thresh = bds[1] + 0.7 * (bds[3] - bds[1])
scot_clip = sh.geometry.box(bds[0] - 10000, northing_thresh, bds[2] + 10000, bds[3] + 10000)
scot_border = north_union.boundary.intersection(scot_clip)
parish_df['distScot'] = parish_df.geometry.centroid.distance(scot_border)

# Truncate variable names for Shapefile compatibility
truncDict = {'copyhold_count_1850':'copys_1850', 'copyhold_count':'copys', 'NrGentry_1400':'gent_1400', 'copyhold_count_1516':'copys_1516', 'perc_catholics_1800':'cath_1800', 'ind_share_1831':'ind_1831', 'agr_share_1831':'agr_1831', 'LS_pc_change':'LS_pc_ch', 'pc_change1525_1086':'pc15251086', 'pc_change1525_1066':'pc15251066', 'pc_change1332_1086':'pc13321086', 'pc_change1332_1066':'pc13321066', 'pc_change1086_1066':'pc10861066', 'LStax_pc_1332':'lspc1332', 'agr_share_1370':'agsh1370', 'ind_share_1370':'indsh1370', 'mean_elevation':'mean_elev', 'wheatsuitability':'wheatsuit', 'distancetoriver':'distriver', 'distancetomarkettown':'distmkt', 'distancetoborder':'distborder', 'distancetolondon':'distlond', 'distancetocoal':'distcoal'}
parish_df = parish_df.rename(columns=truncDict)

Adding final variables...
Joining terrain data...
Joining population data...
Calculating distance to Scottish border...


In [8]:
# --- Terrain land-use area shares ---
print('Computing terrain land-use area shares (pct_arable, pct_pastoral, pct_ag_land)...')

# Classification of each T_ZONE into arable / pastoral / non_ag.
# Based on expected land use in 16th-century northern England; see TerrainZonesDescriptions.csv.
ZONE_USE = {
    'Alluvial plains and river terraces':           'arable',
    'Carboniferous limestone landscapes':           'pastoral',
    'Chalk landscapes':                             'pastoral',
    'Chalky drift veneered plateaux':               'arable',
    'Clay or marl lowlands':                        'arable',
    'Dissected hills':                              'pastoral',
    'Dissected low plateaux and ridges':            'pastoral',
    'Drift veneered dissected hills':               'pastoral',
    'Fenlands':                                     'non_ag',
    'Inland water':                                 'non_ag',
    'Jurassic limestone landscapes':                'pastoral',
    'Lacustrine clay plains':                       'pastoral',
    'Landscapes smothered with deep Chalky drift':  'arable',
    'Landscapes smothered with deep Red drift':     'arable',
    'Landscapes smothered with other deep drift':   'arable',
    'Loess deposits':                               'arable',
    'Magnesian limestone landscapes':               'pastoral',
    'Marshes':                                      'non_ag',
    'Mountains, plateaux and dissected plateaux':   'non_ag',
    'Other drift veneered plateaux':                'arable',
    'Outwash sands and gravels':                    'pastoral',
    'Red drift veneered plateaux':                  'arable',
    'Sandstone and sandy lands':                    'pastoral',
    'Sandstone escarpments and ridges':             'non_ag',
    'Sandy lands with some clays and gravels':      'pastoral',
    'Terminal moraines and drumlins':               'pastoral',
    'Upland valley lands and dissected plateaux':   'pastoral',
}

# Tag each terrain polygon with its land-use category
tz = terrain_df[['T_ZONE', 'geometry']].copy()
tz['land_use'] = tz['T_ZONE'].map(ZONE_USE).fillna('non_ag')

# Reset index so par_county becomes a column for the overlay join
parish_reset = parish_df[['geometry']].reset_index()   # par_county column + geometry

# Intersect parishes with terrain zones to get exact overlap polygons
terrain_ovl = gp.overlay(
    parish_reset[['par_county', 'geometry']],
    tz[['land_use', 'geometry']],
    how='intersection',
    keep_geom_type=False,
)
terrain_ovl['ovl_area'] = terrain_ovl.geometry.area   # m²

# Parish total areas (m²), indexed by par_county
parish_area_m2 = parish_df.geometry.area

# Sum overlap area by parish x land-use category, then pivot
area_sums = (
    terrain_ovl
    .groupby(['par_county', 'land_use'])['ovl_area']
    .sum()
    .unstack(fill_value=0.0)
)

# Ensure all three categories are present even if absent from the north
for col in ['arable', 'pastoral', 'non_ag']:
    if col not in area_sums.columns:
        area_sums[col] = 0.0

# Divide by parish area to get fractional shares
area_pct = area_sums.div(parish_area_m2, axis=0)
area_pct['pct_arable']   = area_pct['arable']
area_pct['pct_pastoral']  = area_pct['pastoral']
area_pct['pct_ag_land']   = area_pct['arable'] + area_pct['pastoral']

parish_df = parish_df.join(area_pct[['pct_arable', 'pct_pastoral', 'pct_ag_land']])
print(f"Terrain shares computed. Mean pct_ag_land: {parish_df['pct_ag_land'].mean():.3f}")


Computing terrain land-use area shares (pct_arable, pct_pastoral, pct_ag_land)...
Terrain shares computed. Mean pct_ag_land: 0.921


## 3. Final Variable Processing (Logic from 02_processing.py)

In [9]:
print('--- Step 3: Final Variable Creation and Cleaning ---')
pdf = parish_df.copy()
pdf['muster'] = pdf['muster'].replace({2: 1, 3: 1})
pdf['primary'] = pdf['primary'].replace({2: 1, 3: 1})

# Generate terrain and county dummies
terrainDummies = pd.get_dummies(pdf['terrainTyp'], drop_first=True).astype(int)
terrainDummies.columns = [c.lower() for c in terrainDummies.columns]
pdf = pd.concat([pdf, terrainDummies], axis=1)

countyDummies = pd.get_dummies(pdf['county'], drop_first=True).astype(int)
pdf = pd.concat([pdf, countyDummies], axis=1)

# Logarithmic transformations and derived metrics
pdf['nonDissLand'] = pdf['landOwned'] - pdf['dissLand']
pdf['ldissLand'] = np.log(pdf['dissLand'] + 1)
pdf['lnonDissLand'] = np.log(pdf['nonDissLand'] + 1)
pdf['bigLand'] = pdf['landOwned'] - pdf['smLand']
pdf['lsmLand'] = np.log(pdf['smLand'] + 1)
pdf['lbigLand'] = np.log(pdf['bigLand'] + 1)
pdf['lsmTithe'] = np.log(pdf['smTithe'] + 1)
pdf['lbgTithe'] = np.log(pdf['bgTithe'] + 1)
pdf['titheIncCalc'] = pdf['titheOutTot'] * 10
pdf['ltitheIncCalc'] = np.log(pdf['titheIncCalc'] + 1)
pdf['tithed'] = (pdf['titheIncCalc'] > 0).astype(int)
pdf['landOwnedShare'] = (pdf['landOwned'] / pdf['titheIncCalc']).replace([np.nan, np.inf], [0, 1000])
pdf.loc[pdf['landOwnedShare'] > 1, 'tithed'] = 0
pdf['smOwnLand'] = 0
pdf.loc[pdf['smHouse'] == 1, 'smOwnLand'] = pdf['ownLandVal']
pdf['smOtherLand'] = pdf['smLand'] - pdf['smOwnLand']
pdf['lpopC'] = np.log(pdf['popC'] + 1)
pdf['lnetInc'] = np.log(pdf['rhNetInc'] + 1)
pdf['llandOwned'] = np.log(pdf['landOwned'] + 1)
pdf['day'] = pdf['day'].fillna(40).astype(int)
pdf['lotherLand'] = np.log(pdf['otherLandVal'] + 1)
pdf['lownLand'] = np.log(pdf['ownLandVal'] + 1)


for monast_var in ['land', 'alms', 'tithe']:
    pdf[f'l{monast_var}InTot'] = np.log(pdf[f'{monast_var}InTot'] + 1)
    pdf[f'l{monast_var}OutTot'] = np.log(pdf[f'{monast_var}OutTot'] + 1)

# Per capita and per square km denominator versions of monastic variables
safe_popC = pdf['popC'].replace(0, np.nan)
safe_area = pdf['area'].replace(0, np.nan)  # area is in km² (divided by 1M above)

pdf['lo_pc'] = pdf['landOwned'] / safe_popC
pdf['lo_sk'] = pdf['landOwned'] / safe_area
safe_ag_area = (pdf['area'] * pdf['pct_ag_land']).replace(0, np.nan)  # agricultural area in km²
pdf['lo_sak'] = pdf['landOwned'] / safe_ag_area
safe_ar_area = (pdf['area'] * pdf['pct_arable']).replace(0, np.nan)  # arable area in km²
pdf['lo_arak'] = pdf['landOwned'] / safe_ar_area
pdf['sm_pc'] = pdf['smLand'] / safe_popC
pdf['sm_sk'] = pdf['smLand'] / safe_area
pdf['bg_pc'] = pdf['bigLand'] / safe_popC
pdf['bg_sk'] = pdf['bigLand'] / safe_area
pdf['st_pc'] = (pdf['smTithe'] * 10) / safe_popC
pdf['st_sk'] = (pdf['smTithe'] * 10) / safe_area
pdf['bt_pc'] = (pdf['bgTithe'] * 10) / safe_popC
pdf['bt_sk'] = (pdf['bgTithe'] * 10) / safe_area
pdf['ti_pc'] = pdf['titheIncCalc'] / safe_popC
pdf['ti_sk'] = pdf['titheIncCalc'] / safe_area
pdf['al_sk'] = pdf['almsInTot'] / safe_area
pdf['own_sk'] = pdf['ownLandVal'] / safe_area
pdf['oth_sk'] = pdf['otherLandVal'] / safe_area
pdf['ni_pc'] = pdf['rhNetInc'] / safe_popC
pdf['ni_sk'] = pdf['rhNetInc'] / safe_area

# Per arable area and per agricultural area: smLand, bigLand, titheOutTot, almsInTot
pdf['sm_sak'] = pdf['smLand'] / safe_ag_area
pdf['bg_sak'] = pdf['bigLand'] / safe_ag_area
pdf['st_sak'] = pdf['smTithe'] / safe_ag_area
pdf['bt_sak'] = pdf['bgTithe'] / safe_ag_area
pdf['ti_sak'] = pdf['titheOutTot'] / safe_ag_area
pdf['al_sak'] = pdf['almsInTot'] / safe_ag_area
pdf['sm_arak'] = pdf['smLand'] / safe_ar_area
pdf['bg_arak'] = pdf['bigLand'] / safe_ar_area
pdf['st_arak'] = pdf['smTithe'] / safe_ar_area
pdf['bt_arak'] = pdf['bgTithe'] / safe_ar_area
pdf['ti_arak'] = pdf['titheOutTot'] / safe_ar_area
pdf['al_arak'] = pdf['almsInTot'] / safe_ar_area
pdf['ni_arak'] = pdf['rhNetInc'] / safe_ar_area
pdf['own_arak'] = pdf['ownLandVal'] / safe_ar_area
pdf['oth_arak'] = pdf['otherLandVal'] / safe_ar_area

# Distance-weighted per capita and per sq km
pdf['lo_dwpc'] = pdf['lo_dw'] / safe_popC
pdf['lo_dwsk'] = pdf['lo_dw'] / safe_area
pdf['sl_dwpc'] = pdf['sl_dw'] / safe_popC
pdf['sl_dwsk'] = pdf['sl_dw'] / safe_area
pdf['bl_dwpc'] = pdf['bl_dw'] / safe_popC
pdf['bl_dwsk'] = pdf['bl_dw'] / safe_area
pdf['ti_dwpc'] = pdf['ti_dw'] / safe_popC
pdf['ti_dwsk'] = pdf['ti_dw'] / safe_area

# Note: llo_dw, lsl_dw, lbl_dw, lti_dw are already log-transformed and distance-weighted from line-level aggregation
# No further transformation needed
pdf['llo_pc'] = np.log(pdf['lo_pc'].fillna(0) + 1)
pdf['llo_sk'] = np.log(pdf['lo_sk'].fillna(0) + 1)
pdf['llo_sak'] = np.log(pdf['lo_sak'].fillna(0) + 1)
pdf['llo_arak'] = np.log(pdf['lo_arak'].fillna(0) + 1)
pdf['lsm_pc'] = np.log(pdf['sm_pc'].fillna(0) + 1)
pdf['lsm_sk'] = np.log(pdf['sm_sk'].fillna(0) + 1)
pdf['lbg_pc'] = np.log(pdf['bg_pc'].fillna(0) + 1)
pdf['lbg_sk'] = np.log(pdf['bg_sk'].fillna(0) + 1)
pdf['lst_pc'] = np.log(pdf['st_pc'].fillna(0) + 1)
pdf['lst_sk'] = np.log(pdf['st_sk'].fillna(0) + 1)
pdf['lbt_pc'] = np.log(pdf['bt_pc'].fillna(0) + 1)
pdf['lbt_sk'] = np.log(pdf['bt_sk'].fillna(0) + 1)
pdf['lti_pc'] = np.log(pdf['ti_pc'].fillna(0) + 1)
pdf['lti_sk'] = np.log(pdf['ti_sk'].fillna(0) + 1)
pdf['lal_sk'] = np.log(pdf['al_sk'].fillna(0) + 1)
pdf['lown_sk'] = np.log(pdf['own_sk'].fillna(0) + 1)
pdf['loth_sk'] = np.log(pdf['oth_sk'].fillna(0) + 1)
pdf['lni_sk'] = np.log(pdf['ni_sk'].fillna(0) + 1)
pdf['lsm_sak'] = np.log(pdf['sm_sak'].fillna(0) + 1)
pdf['lbg_sak'] = np.log(pdf['bg_sak'].fillna(0) + 1)
pdf['lst_sak'] = np.log(pdf['st_sak'].fillna(0) + 1)
pdf['lbt_sak'] = np.log(pdf['bt_sak'].fillna(0) + 1)
pdf['lti_sak'] = np.log(pdf['ti_sak'].fillna(0) + 1)
pdf['lal_sak'] = np.log(pdf['al_sak'].fillna(0) + 1)
pdf['lsm_arak'] = np.log(pdf['sm_arak'].fillna(0) + 1)
pdf['lbg_arak'] = np.log(pdf['bg_arak'].fillna(0) + 1)
pdf['lst_arak'] = np.log(pdf['st_arak'].fillna(0) + 1)
pdf['lbt_arak'] = np.log(pdf['bt_arak'].fillna(0) + 1)
pdf['lti_arak'] = np.log(pdf['ti_arak'].fillna(0) + 1)
pdf['lal_arak'] = np.log(pdf['al_arak'].fillna(0) + 1)
pdf['lni_arak'] = np.log(pdf['ni_arak'].fillna(0) + 1)
pdf['lown_arak'] = np.log(pdf['own_arak'].fillna(0) + 1)
pdf['loth_arak'] = np.log(pdf['oth_arak'].fillna(0) + 1)
# Per capita and per sq km logged distance-weighted variables
pdf['llo_dwpc'] = pdf['llo_dw'] / safe_popC
pdf['llo_dwsk'] = pdf['llo_dw'] / safe_area
pdf['lsl_dwpc'] = pdf['lsl_dw'] / safe_popC
pdf['lsl_dwsk'] = pdf['lsl_dw'] / safe_area
pdf['lbl_dwpc'] = pdf['lbl_dw'] / safe_popC
pdf['lbl_dwsk'] = pdf['lbl_dw'] / safe_area
pdf['lti_dwpc'] = pdf['lti_dw'] / safe_popC
pdf['lti_dwsk'] = pdf['lti_dw'] / safe_area

# Final Output
# pdf.drop(columns=['par_county'], inplace=True)  # Drop ID for shapefile compatibility
print(f'Saving final output to {PROCESSED / "northParishFlows.shp"}')
pdf.to_file(PROCESSED / 'northParishFlows.shp')
print('DONE!')

--- Step 3: Final Variable Creation and Cleaning ---
Saving final output to c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\northParishFlows.shp
DONE!

c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\.venv\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\.venv\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\.venv\Lib\site-packages\geopandas\geodataframe.py:1969: PerformanceWarning: Data